In [1]:
import networkx as nx
from yfiles_jupyter_graphs import GraphWidget
from graphrag.index.operations.cluster_graph import cluster_graph
import pandas as pd

# converts the entities dataframe to a list of dicts for yfiles-jupyter-graphs
def convert_entities_to_dicts(nodes, df, level=0):
    """Convert the entities dataframe to a list of dicts for yfiles-jupyter-graphs."""
    nodes_dict = {}
    for node in nodes:
        # Create a dictionary for each row and collect unique nodes
        community_df = df[(df['title']==node['properties']['label']) & (df["level"]==level)]
        if len(community_df) > 0:
            community_info = community_df.iloc[0].to_dict()
        else:
            community_info = {"community": None, "title": node['properties']['label']}
        node["properties"].update(community_info)
    return nodes

# map community to a color
def community_to_color(community):
    """Map a community to a color."""
    colors = [
        "crimson",
        "darkorange",
        "indigo",
        "cornflowerblue",
        "cyan",
        "teal",
        "green",
        "gold",
        "brown",
    ]
    return (
        colors[int(community) % len(colors)] if community is not None else "lightgray"
    )


def edge_to_source_community(edge):
    """Get the community of the source node of an edge."""
    source_node = next(
        (entry for entry in w.nodes if entry["id"] == edge["start"]),
        None,
    )
    source_node_community = source_node["properties"]["community"]
    return source_node_community if source_node_community is not None else None


# LightRAG の生成物を可視化するため、現在は `dickens` を参照する。
dataset = "dickens"
filepath = f'./{dataset}/graph_chunk_entity_relation.graphml'

# GraphWidget インスタンスの作成
G = nx.read_graphml(filepath)
w = GraphWidget(graph=G)

# graphrag==3.0.8 の cluster_graph は networkx.Graph ではなく
# source / target 列を持つ DataFrame を受け取る。
edges = pd.DataFrame([
    {"source": source, "target": target, **attrs}
    for source, target, attrs in G.edges(data=True)
])

# クラスタリングを実行
max_cluster_size = 10  # クラスタの最大サイズ
use_lcc = True         # 最大全結合成分のみを使用
seed = 0xDEADBEEF      # ランダムシード

communities = cluster_graph(edges, max_cluster_size=max_cluster_size, use_lcc=use_lcc, seed=seed)
base_communities = pd.DataFrame(
  communities, columns=pd.Index(["level", "community", "parent", "title"])
).explode("title")
base_communities["community"] = base_communities["community"].astype(int)

w.directed = True
w.nodes = convert_entities_to_dicts(w.get_nodes(), base_communities, level=0)

# show title on the node
w.node_label_mapping = "label"
w.node_color_mapping = lambda node: community_to_color(node["properties"]["community"])
w.edge_color_mapping = lambda edge: community_to_color(edge_to_source_community(edge))
# use weight for edge thickness
w.edge_thickness_factor_mapping = "weight"

# 結果を表示
w.circular_layout()
w.show()

GraphWidget(layout=Layout(height='800px', width='100%'))

In [2]:
display(w.get_nodes()[0])
print("="*200)
display(base_communities[base_communities["title"]=='"HTML"'])
print("="*200)
display(w.get_edges()[0])

{'id': 0,
 'properties': {'entity_id': 'The Mother',
  'entity_type': 'person',
  'description': 'The mother is responsible for laying work upon a table, commenting that candle-light makes her eyes hurt, and remaining cheerful while working and speaking to her family.',
  'source_id': 'chunk-1d4b58de5429cd1261370c231c8673e8',
  'file_path': 'C:\\Users\\09027\\Documents\\05.portforio\\書籍\\20260409_GraphRAG完全ガイド\\graphrag-demo\\book.txt',
  'created_at': 1775752147,
  'truncate': '',
  'label': 'The Mother',
  'community': None,
  'title': 'The Mother'},
 'color': 'lightgray',
 'styles': {},
 'label': 'The Mother',
 'scale_factor': 1.0,
 'type': 'lightgray',
 'size': (55.0, 55.0),
 'position': (0.0, 0.0)}

,level,community,parent,title


{'id': 0,
 'start': 0,
 'end': 9,
 'properties': {'weight': 1.0,
  'description': 'The mother places her work upon the table.',
  'keywords': 'household work,preparation',
  'source_id': 'chunk-1d4b58de5429cd1261370c231c8673e8',
  'file_path': 'C:\\Users\\09027\\Documents\\05.portforio\\書籍\\20260409_GraphRAG完全ガイド\\graphrag-demo\\book.txt',
  'created_at': 1775752247,
  'truncate': ''},
 'color': 'lightgray',
 'thickness_factor': 1.0,
 'directed': True,
 'styles': {},
 'label': ''}